# nmtc-application-builder — Full Application Walkthrough

**Week 2 of 4: Section Generators, Table Builders & Output Renderers**

This notebook demonstrates the complete Week 2 workflow.
A CDE can produce a competition-ready application package with a single `generate()` call.

1. Build a CDE profile and pipeline
2. Run `app.analyze()` for intelligence
3. Inspect section generator output (Sections A-E)
4. Inspect table builder output (6 supporting tables)
5. Call `app.generate()` for Word, Excel, PDF, and Markdown
6. Verify the generated files

>  All calls work offline. External APIs fall back to sample data automatically.

In [ ]:
import sys, os
sys.path.insert(0, '..')

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline
import pandas as pd

print("nmtc-application-builder imports OK")

## 1. Build CDE Profile and Pipeline

In [ ]:
cde = CDEProfile.sample()
pipeline = Pipeline.sample(n=20)

app = Application(
    cde=cde,
    requested_allocation=65_000_000,
    application_round="CY2025",
)
app.add_pipeline(pipeline)

print(f"CDE:        {cde.name}")
print(f"Pipeline:   {len(pipeline)} projects")
print(f"Total QEI:  ${sum(p.qei_request for p in pipeline):,.0f}")

## 2. Run Analysis

In [ ]:
analysis = app.analyze()
score = analysis.readiness_score

print(f"Readiness Score:      {score.overall_score:.1f}/100 (Grade {score.grade})")
print(f"Deep/Severe Distress: {analysis.distress_analysis['pct_deep_or_severe']:.0%}")
print(f"States:               {analysis.geographic_analysis['states_count']}")
print(f"Jobs Created:         {analysis.impact_summary['total_jobs_created']:,}")

## 3. Section Generator Output

In [ ]:
from nmtcapp.sections import ALL_SECTIONS

for section_gen in ALL_SECTIONS:
    content = section_gen.generate_content(app, analysis)
    n_subs = len(content["subsections"])
    print(f"Section {content['section_id']}: {content['title']}  ({n_subs} subsections)")

### Section A — Investment Thesis Narrative

In [ ]:
from nmtcapp.sections.section_a_business import SectionABusinessStrategy

content_a = SectionABusinessStrategy().generate_content(app, analysis)
thesis = next(s for s in content_a["subsections"] if "thesis" in s["heading"].lower())
print(f"### {thesis['heading']}\n")
print(thesis["body"][:600], "...")

### Section A — Markdown Rendering

In [ ]:
from nmtcapp.sections.section_a_business import SectionABusinessStrategy

md_section = SectionABusinessStrategy().generate_markdown(app, analysis)
print(md_section[:800])

## 4. Table Builder Output

### Pipeline Table (Appendix A)

In [ ]:
from nmtcapp.tables.pipeline_table import build_pipeline_table

df_pipeline = build_pipeline_table(pipeline, cde)
print(f"Shape: {df_pipeline.shape}")
cols = ["Project Name", "State", "Sector (NAICS)", "QEI Request ($)", "Distress Level", "NMTC Eligible (Y/N)"]
display(df_pipeline[cols].head(8))

### Distress Documentation (Appendix B)

In [ ]:
from nmtcapp.tables.distress_table import build_distress_table

df_distress = build_distress_table(pipeline)
exclude = {"ACS Vintage", "CDFI Fund Source"}
display(df_distress[[c for c in df_distress.columns if c not in exclude]].head(6))

### Geographic Targeting (Appendix C)

In [ ]:
from nmtcapp.tables.geographic_table import build_geographic_table

df_geo = build_geographic_table(pipeline)
display(df_geo.head(10))

### Impact Projections (Appendix D)

In [ ]:
from nmtcapp.tables.impact_table import build_impact_table

df_impact = build_impact_table(pipeline)
cols = ["Project Name", "State", "Sector (NAICS)", "Jobs Created", "QEI Request ($)", "Cost Per Job ($)"]
display(df_impact[cols].head(8))

## 5. Generate Full Application Package

In [ ]:
output_dir = "../examples/sample_output"

paths = app.generate(
    output_dir=output_dir,
    formats=["markdown", "word", "excel", "pdf"],
)

print("Generated files:")
for fmt, path in paths.items():
    size_kb = os.path.getsize(path) // 1024
    print(f"  {fmt:10s}  {os.path.basename(path)}  ({size_kb} KB)")

## 6. Verify Generated Files

### Markdown — first 1500 chars

In [ ]:
with open(paths["markdown"]) as f:
    md_content = f.read()

print(md_content[:1500])

### Excel — sheet inventory

In [ ]:
import openpyxl

wb = openpyxl.load_workbook(paths["excel"])
print("Excel sheets:")
for name in wb.sheetnames:
    ws = wb[name]
    print(f"  {name:30s}  ({ws.max_row} rows x {ws.max_column} cols)")

### Word — paragraph and table count

In [ ]:
from docx import Document

word_doc = Document(paths["word"])
print(f"Word document: {len(word_doc.paragraphs)} paragraphs, {len(word_doc.tables)} tables")

### PDF — validity check

In [ ]:
with open(paths["pdf"], "rb") as f:
    header = f.read(4)

size_kb = os.path.getsize(paths["pdf"]) // 1024
print(f"PDF header: {header}  (valid PDF: {header == b'%PDF'})")
print(f"PDF size:   {size_kb} KB")

## Summary

With a single `app.generate()` call, the library produces:

| Format | Contents |
|--------|----------|
| **Markdown** | Full application draft, version-control friendly |
| **Word** | Professional `.docx` with cover, sections A-E, 6 appendices |
| **Excel** | 7-sheet workbook with conditional formatting and charts |
| **PDF** | Board-ready PDF via ReportLab |

Each document includes:
- Cover page with CDE branding and readiness grade
- Executive summary with key metrics table
- Sections A-E with narrative, tables, and bullet lists
- Appendices: pipeline, distress, geographic, impact, track record, methodology

**Week 3** will add: win probability scoring, optimizer, HMDA integration, and Plotly visualizations.

See the [GitHub repo](https://github.com/Jaypatel1511/nmtc-application-builder) for the roadmap.